# StudyMate: освітній асистент з точних наук

**Фінальний проєкт курсу AI Fundamental**
**Автор:** Яковенко Сергій

---

## Одним абзацом

StudyMate допомагає студенту, який готується до іспиту сам, знайти не просто формулу,
а **правильну для його ситуації** формулу. Ключова властивість системи не в тому,
що вона багато знає, а в тому, що вона **не вигадує**: усі факти приходять з перевіреної
бази, придатність формули перевіряє код, а коли даних немає, система про це прямо каже.

## Чому саме так

Цей принцип не взятий з підручника, я прийшов до нього через два власні результати.

**Експеримент з ембеддінгами.** Я чисельно перевірив, як модель бачить два речення:
«формула працює лише коли точка кидання і точка падіння на одній висоті» і «камінь
кидають з даху, тому початкова висота не дорівнює нулю». Логічно це пряма суперечність,
друге описує ситуацію, у якій перше забороняє застосовувати формулу. Для моделі вони
просто близькі за темою. **Ембеддінг кодує тему, а не істинність.**

**Баг у власному пошуку.** Перша версія ранжування формул була односторонньою мірою:
скільки слів назви знайшлося в запиті. На запиті «закон збереження енергії» вона
повернула **ЗАКОН ОМА з оцінкою 1.0**, тобто як ідеальний збіг. Причина дрібна: слово
«ома» коротке, фільтр його викидав, у назві лишалося одне слово «закон», воно в запиті є.

Обидва випадки це одна й та сама помилка: **впевнена неправильна відповідь**. Для
освітнього продукту вона небезпечніша за відмову, бо студент звернувся саме тому,
що не може її перевірити. Уся архітектура нижче побудована навколо цього.

## Крок 0. Середовище

Код системи живе не в ноутбуці, а в пакеті `studymate`, розбитому за
відповідальностями. Ноутбук його **використовує**, а не дублює: інакше дві копії
однієї логіки неминуче розійшлися б, і демо показувало б не те, що працює
у веб-інтерфейсі.

| Модуль | Відповідальність |
|---|---|
| `models.py` | типи даних, без логіки й без залежностей |
| `data.py` | довідник формул і таблиці перетворень |
| `text.py` | стемінг і міра схожості, чисті функції |
| `search.py` | лексичний і семантичний шари, злиття через RRF |
| `applicability.py` | фільтр застосовності: рішення ухвалює код, не модель |
| `converters.py` | переведення одиниць |
| `planner.py` | планування підготовки |
| `tools.py` | тонкі `@tool`-обгортки над готовими шарами |
| `agent.py` | агент, системний промпт і діалог з контекстом |

In [1]:
!pip install --quiet "langchain>=1.0" "langchain-openai>=1.0" langgraph pandas numpy

zsh:1: command not found: pip


In [2]:
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO = "https://github.com/Srh-Yakovenko-ua/AI_FUNDAMENTAL_FINAL.git"

# Пакет беремо з репозиторію проєкту: у Colab його ще немає, тому клонуємо.
# Через subprocess, а не через ! shell-магію: так комірка лишається звичайним
# Python і працює однаково в Colab, локально та в тестах.
if not os.path.exists("studymate") and not os.path.exists("AI_FUNDAMENTAL_FINAL"):
    subprocess.run(["git", "clone", "-q", REPO], check=False)
if os.path.isdir("AI_FUNDAMENTAL_FINAL"):
    sys.path.insert(0, "AI_FUNDAMENTAL_FINAL")

import studymate

print("✅ Пакет studymate завантажено")
print(f"   Формул у базі: {len(studymate.FORMULAS)}")
print(f"   З умовами застосовності: {sum(1 for f in studymate.FORMULAS if f.predicates)}")
print(f"   Інструментів: {len(studymate.TOOLS)}")

✅ Пакет studymate завантажено
   Формул у базі: 12
   З умовами застосовності: 6
   Інструментів: 4


In [3]:
# Ключ беремо з Colab Secrets, далі зі змінної середовища, і лише потім питаємо вручну.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("✅ Ключ завантажено з Colab Secrets")
    except Exception:
        import getpass

        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
        print("✅ Ключ встановлено вручну")
else:
    print("✅ Ключ узято зі змінної середовища")

✅ Ключ узято зі змінної середовища


In [4]:
from studymate import (FORMULAS, SEARCH, StudyMateAgent, check_formula_for_task,
                       convert_units, detect_conditions, formula_lookup, plan_exam_prep)

In [5]:
# Семантичний шар вмикається лише тут: імпорт пакета лишається дешевим
# і не потребує ані ключа, ані мережі.
semantic_ready = SEARCH.build()
print(f"{'✅' if semantic_ready else '⚠️ '} Семантичний шар: "
      f"{'увімкнено' if semantic_ready else 'вимкнено, працюємо на лексичному пошуку'}")

agent = StudyMateAgent(model_name="openai/gpt-oss-120b")
print(f"✅ Агент: {agent.model_name}, temperature={agent.temperature}, "
      f"max_tokens={agent.max_tokens}")


def ask(text, history=None, verbose=True):
    """Тонка обгортка для ноутбука: повертає (відповідь, історія, інструменти)."""
    turn = agent.ask(text, history, verbose=verbose)
    return turn.answer, turn.history, turn.tools_used


ERROR_PREFIX = studymate.ERROR_PREFIX

⚠️  Семантичний шар: вимкнено, працюємо на лексичному пошуку
✅ Агент: openai/gpt-oss-120b, temperature=0.2, max_tokens=900


### Про модель, на якій виконано це демо

Прогін зроблено на `openai/gpt-oss-120b` через OpenAI-сумісний ендпоінт Groq: ключа
OpenAI під рукою не було, а безкоштовний тариф Groq дає ту саму API. Архітектура від
цього не змінилась, клас лишився `ChatOpenAI`, підмінено тільки `base_url`.

Два наслідки видно прямо у виводах вище, і обидва передбачені конструкцією:

1. **Семантичний шар вимкнувся.** Ембеддінги `text-embedding-3-small` існують лише в
   OpenAI, тому `SEARCH.build()` повернув `False`, і система чесно повідомила, що
   працює на лексичному пошуку. Це рівно та деградація, яка закладалась: продукт не
   падає без семантики, він втрачає якість і каже про це вголос.
2. **Ціни в розділі вартості це прайс `gpt-4o-mini`.** Лічильники токенів справжні, з
   `usage_metadata` фактичних відповідей, але помножені на прайс цільової моделі, а не
   на рахунок від Groq. Тобто це перерахунок під модель, для якої продукт
   проєктувався, і саме так його треба читати.


## Крок 1. Дані: база формул з умовами застосовності

Серце системи. Кожна картка несе три речі, яких немає ні в пошуковій видачі,
ні в пам'яті моделі: машиночитані `predicates` з умовами застосовності,
`synonyms` з тим, як формулу називає студент, і `note` з попередженням,
яке обов'язково доходить до нього.

In [6]:
print("ДОВІДНИК ФОРМУЛ\n")
display(pd.DataFrame([
    {"Предмет": f.subject, "Формула": f.name, "Вираз": f.expression,
     "Умови застосовності": ", ".join(f.predicates) if f.predicates else "немає",
     "Синонімів": len(f.synonyms)}
    for f in FORMULAS
]))

ДОВІДНИК ФОРМУЛ



,Предмет,Формула,Вираз,Умови застосовності,Синонімів
0,фізика,"дальність польоту тіла, кинутого під кутом",L = v₀² · sin(2α) / g,"h0 == 0, air_resistance == False",3
1,фізика,дальність польоту при киданні з висоти,час t з рівняння h₀ + v₀·sin(α)·t − g·t²/2 = 0...,"h0 > 0, air_resistance == False",3
2,фізика,кінетична енергія,Eₖ = m·v² / 2,v_units == 'м/с',1
3,фізика,потенціальна енергія,Eₚ = m·g·h,немає,1
4,фізика,закон ома,I = U / R,resistance_constant == True,1
5,фізика,середня швидкість,v = s / t,немає,0
6,хімія,рівняння стану ідеального газу,p·V = n·R·T,"T_units == 'К', V_units == 'м³'",2
7,хімія,молярна концентрація,C = n / V,немає,2
8,хімія,молярна маса,M = m / n,немає,0
9,математика,площа кола,S = π·r²,немає,0


## Крок 2. Пошук: гібридний, з явною деградацією

Лексичний шар працює завжди, без мережі й без ключа: ловить назви, символи
і синоніми. Метрика двостороння (F2), де повнота покриття запиту важить більше
за покриття назви. Односторонній варіант я вже пробував, і саме він повертав
закон Ома на запит про збереження енергії.

Семантичний шар додається за наявності ключа і ловить перефразування. Злиття
через RRF: складаються ранги, а не оцінки, тому шкали шарів не треба зводити докупи.

In [7]:
rows = []
for query in ["кінетична енергія", "формула закону ома", "концентрація розчину",
              "кидання з даху", "закон збереження енергії", "швидкість світла"]:
    found = SEARCH.search(query, top_k=1)
    top = found[0] if found else None
    rows.append({
        "Запит": query,
        "Знайдено": top.formula.name if top else "нічого",
        "Лексична оцінка": round(top.lexical_score, 3) if top else 0,
        "Шар": top.found_by if top else "немає",
        "Впевнено": "так" if top and top.confident else "ні",
    })
print("ЯК ПРАЦЮЄ ПОШУК\n")
display(pd.DataFrame(rows))

ЯК ПРАЦЮЄ ПОШУК



,Запит,Знайдено,Лексична оцінка,Шар,Впевнено
0,кінетична енергія,кінетична енергія,1.000,lexical,так
1,формула закону ома,закон ома,1.000,lexical,так
2,концентрація розчину,молярна концентрація,1.000,lexical,так
3,кидання з даху,дальність польоту при киданні з висоти,1.000,lexical,так
4,закон збереження енергії,кінетична енергія,0.357,lexical,ні
5,швидкість світла,середня швидкість,0.500,lexical,ні


## Крок 3. Фільтр застосовності: рішення ухвалює код

Головний компонент системи, і він принципово **не використовує модель**.
Retrieval відповідає на питання «про що це». На питання «чи можна це застосувати
саме тут» він відповісти не здатний: у векторному просторі «умова виконується»
і «умова порушена» лежать поруч.

In [8]:
print("РОЗПІЗНАВАННЯ УМОВ ЗАДАЧІ\n")
display(pd.DataFrame([
    {"Умова задачі": s[:58],
     "Розпізнано": ", ".join(sorted(detect_conditions(s))) or "ознак немає"}
    for s in [
        "Камінь кидають з даху висотою 12 м під кутом 30 градусів",
        "Тіло кидають з рівної поверхні під кутом 45 градусів",
        "Кидаю не з балкона третього поверху, а з землі",
        "Кидання без тертя з даху висотою 12 м",
        "Трикутник не прямокутний, сторони 5, 6, 7",
    ]
]))

РОЗПІЗНАВАННЯ УМОВ ЗАДАЧІ



,Умова задачі,Розпізнано
0,Камінь кидають з даху висотою 12 м під кутом 3...,h0 > 0
1,Тіло кидають з рівної поверхні під кутом 45 гр...,h0 == 0
2,"Кидаю не з балкона третього поверху, а з землі",h0 == 0
3,Кидання без тертя з даху висотою 12 м,h0 > 0
4,"Трикутник не прямокутний, сторони 5, 6, 7",triangle_type != 'прямокутний'


## Крок 6. Демонстрація: три типи поведінки системи

Демо навмисно показує не тільки успіх. Три блоки нижче це три різні режими:
коли система працює як задумано, коли вона тримає контекст діалогу
і коли впирається у власні межі.

### Сценарій 1. Головний: формула правильна, застосування хибне

Це той самий випадок, з якого починався весь продукт. Студент бере формулу
дальності польоту, підставляє дані задачі про кидання з даху і отримує
відповідь, меншу за правильну в півтора рази. Формула не помилкова,
помилкове її застосування, і зовні цієї різниці не видно.

In [9]:
history = []
_, history, _ = ask(
    "Розвʼязую задачу: камінь кидають з даху висотою 12 м під кутом 30° "
    "зі швидкістю 15 м/с. Хочу взяти формулу дальності польоту. Це правильно?",
    history,
)

👤 Розвʼязую задачу: камінь кидають з даху висотою 12 м під кутом 30° зі швидкістю 15 м/с. Хочу взяти формулу дальності польоту. Це правильно?
🔧 check_formula_for_task
🤖 **Суть:** Так, для вашої задачі треба користуватися формулою «дальність польоту при киданні з висоти». Формула «дальність польоту тіла, кинутого під кутом» підходить лише коли стартова висота = 0, тому її тут не можна застосовувати.

**Пояснення:**  
При киданні з даху (висота \(h_0=12\) м) тіло має вертикальну складову швидкості \(v_0\sin\alpha\) і горизонтальну \(v_0\cos\alpha\). Спочатку визначаємо час польоту \(t\) з рівняння вертикального руху:

\[
h_0 + v_0\sin\alpha \, t - \frac{g t^{2}}{2}=0,
\]

де \(g\approx9{,}81\ \text{м/с}^2\). Розв’язавши це квадратне рівняння (беремо додатний корінь), отримуємо час, протягом якого камінь перебуває в повітрі.

Потім горизонтальна дальність:

\[
L = v_0\cos\alpha \; t .
\]

Тобто спочатку треба знайти \(t\), а вже потім підставити його у формулу для \(L\).

**Приклад (для о

### Сценарій 2. Контекст: уточнення без повторення умови

Наступні два запити не мають сенсу без пам'яті про попередні: у них немає
ані чисел, ані назви формули. Це та цінність діалогу, якої не дає пошуковий рядок.

In [10]:
_, history, _ = ask("А якби я кидав з землі, тоді підійшла б?", history)

👤 А якби я кидав з землі, тоді підійшла б?
🤖 **Суть:** Якщо камінь кидають з рівня землі (початкова висота \(h_0 = 0\)), то підходить формула «дальність польоту тіла, кинутого під кутом». Вона саме виведена для кидання з нульової висоти.

**Пояснення:**  
Для кидання з рівня землі вертикальна складова швидкості спочатку піднімає тіло, а потім воно падає назад до тієї ж висоти, звідки стартувало. Час польоту у цьому випадку можна знайти просто:

\[
t = \frac{2 v_0 \sin\alpha}{g},
\]

а дальність:

\[
L = v_0 \cos\alpha \; t = \frac{v_0^{2}\sin 2\alpha}{g}.
\]

Тобто формула «\(L = \dfrac{v_0^{2}\sin 2\alpha}{g}\)» (або її еквівалент) використовується лише коли стартова висота = 0. Якщо ж висота не нульова, треба спочатку розв’язати квадратне рівняння для часу, як у попередньому випадку. 

Тож:  
- **З даху (h₀ > 0)** – формула «дальність польоту при киданні з висоти».  
- **З землі (h₀ = 0)** – формула «дальність польоту тіла, кинутого під кутом».  

Якщо потрібна допомога з підстановко

In [11]:
_, history, _ = ask("Швидкість у мене в км/год, 54. Переведи, будь ласка.", history)
print(f"Довжина історії діалогу: {len(history)} повідомлень")

👤 Швидкість у мене в км/год, 54. Переведи, будь ласка.
🔧 convert_units
🤖 **Суть:** 54 км/год = **15 м/с**. 

Тепер ви можете підставити цю швидкість у формули, які розглядали. Якщо треба продовжити розрахунок або уточнити інші параметри, дайте знати!
──────────────────────────────────────────────────────────────────────────────
Довжина історії діалогу: 10 повідомлень


### Сценарій 3. Межі: три випадки, де система має зупинитися

In [12]:
# 3.1 Формули немає в базі. Правильна поведінка це відмова, а не згадування з пам'яті.
ask("Яка формула ентропії Гіббса для відкритих систем?", history=[])

👤 Яка формула ентропії Гіббса для відкритих систем?
🔧 formula_lookup
🤖 **Суть:** У базі StudyMate немає запису формули ентропії Гіббса саме для відкритих систем.

**Пояснення:** Я скористався інструментом `formula_lookup`, який шукає лише перевірені формули з нашої бази даних. Запит «ентропія Гіббса відкриті системи» не дав результату, тому я не можу надати точну формулу чи умови її застосовності. Якщо потрібна інша формула (наприклад, для закритих систем або інша термодинамічна величина), уточніть запит, і я спробую знайти відповідну інформацію.
──────────────────────────────────────────────────────────────────────────────


('**Суть:** У базі StudyMate немає запису формули\u202fентропії Гіббса саме для відкритих систем.\n\n**Пояснення:** Я скористався інструментом\u202f`formula_lookup`, який шукає лише перевірені формули з нашої бази даних. Запит «ентропія Гіббса відкриті системи» не дав результату, тому я не можу надати точну формулу чи умови її застосовності. Якщо потрібна інша формула (наприклад, для закритих систем або інша термодинамічна величина), уточніть запит, і я спробую знайти відповідну інформацію.',
 [HumanMessage(content='Яка формула ентропії Гіббса для відкритих систем?', additional_kwargs={}, response_metadata={}, id='c7513313-77a1-417a-afe3-91a1f307a116'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 1271, 'total_tokens': 1347, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 37, 'rejected_prediction_tokens': None}, 'prompt_tokens_details'

In [13]:
# 3.2 Запит неоднозначний. Правильна поведінка це перепитати, а не обрати за студента.
ask("Дай формулу дальності польоту", history=[])

👤 Дай формулу дальності польоту
🔧 formula_lookup
🤖 Уточни, будь ласка, яку саме дальність польоту тебе цікавить:  

* **Дальність польоту при киданні з висоти** (коли тіло стартує з певної висоти і має початкову швидкість горизонтально);  

* **Дальність польоту тіла, кинутого під кутом** (класичний випадок кидка під кутом до горизонту з рівнянням траєкторії).  

Опиши умову задачі або вкажи, який варіант потрібен, і я перевірю, чи підходить обрана формула.
──────────────────────────────────────────────────────────────────────────────


('Уточни, будь ласка, яку саме дальність польоту тебе цікавить:  \n\n* **Дальність польоту при киданні з висоти** (коли тіло стартує з певної висоти і має початкову швидкість горизонтально);  \n\n* **Дальність польоту тіла, кинутого під кутом** (класичний випадок кидка під кутом до горизонту з рівнянням траєкторії).  \n\nОпиши умову задачі або вкажи, який варіант потрібен, і я перевірю, чи підходить обрана формула.',
 [HumanMessage(content='Дай формулу дальності польоту', additional_kwargs={}, response_metadata={}, id='14b61289-5186-47b2-8c6b-92b3ef8c25d5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 1261, 'total_tokens': 1316, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 23, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 1024}, 'queue_time': 0.2056

In [14]:
# 3.3 Питання поза профілем.
ask("Порадь, що приготувати на вечерю з курки", history=[])

👤 Порадь, що приготувати на вечерю з курки
🤖 Вибач, я — StudyMate, освітній асистент, який допомагає лише з питаннями з математики, фізики та хімії. Тому порадити, що приготувати на вечерю з курки, я, на жаль, не можу. Якщо у вас є питання щодо розрахунків, формул чи експериментів у цих предметах — із задоволенням допоможу!
──────────────────────────────────────────────────────────────────────────────


('Вибач, я\u202f—\u202fStudyMate, освітній асистент, який допомагає лише з питаннями з математики, фізики та хімії. Тому порадити, що приготувати на вечерю з курки, я, на жаль, не можу. Якщо у вас є питання щодо розрахунків, формул чи експериментів у цих предметах — із задоволенням допоможу!',
 [HumanMessage(content='Порадь, що приготувати на вечерю з курки', additional_kwargs={}, response_metadata={}, id='8b074951-6119-44d7-a0dd-0adc97e1def5'),
  AIMessage(content='Вибач, я\u202f—\u202fStudyMate, освітній асистент, який допомагає лише з питаннями з математики, фізики та хімії. Тому порадити, що приготувати на вечерю з курки, я, на жаль, не можу. Якщо у вас є питання щодо розрахунків, формул чи експериментів у цих предметах — із задоволенням допоможу!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 1267, 'total_tokens': 1428, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reas

## Крок 7. Автоматичне тестування

Тести перевіряють **дві речі окремо**: який інструмент викликано і що фактично
опинилося у відповіді. Це різні питання: відповідь може бути правильною і при цьому
отриманою з пам'яті моделі, без звернення до бази. Для освітнього продукту саме
походження відповіді і є предметом контролю.

In [15]:
TEST_CASES = [
    {"id": 1, "сценарій": "Базовий пошук формули",
     "запит": "Яка формула кінетичної енергії?",
     "інструмент": "formula_lookup", "має_містити": ["m·v²"]},
    {"id": 2, "сценарій": "Умова застосовності доходить до студента",
     "запит": "Розкажи про формулу для закону Ома",
     "інструмент": "formula_lookup", "має_містити": ["опор"]},
    {"id": 3, "сценарій": "КЛЮЧОВИЙ: перевірка придатності для задачі",
     "запит": "Кидаю камінь з даху 12 м під кутом 30°. Чи можна взяти формулу дальності польоту?",
     "інструмент": "check_formula_for_task", "має_містити": ["висот"]},
    {"id": 4, "сценарій": "Неоднозначний запит",
     "запит": "Дай формулу дальності польоту",
     "інструмент": "formula_lookup", "має_містити": ["уточн"]},
    {"id": 5, "сценарій": "Формули немає в базі",
     "запит": "Яка формула ентропії Гіббса?",
     "інструмент": "formula_lookup", "не_має_містити": ["ΔG =", "G = H"]},
    {"id": 6, "сценарій": "Регресія: чужа формула на схожий запит",
     "запит": "Яка формула закону збереження енергії?",
     "інструмент": "formula_lookup", "не_має_містити": ["I = U / R"]},
    {"id": 7, "сценарій": "Конвертація одиниць",
     "запит": "Скільки кубічних метрів у 40 літрах?",
     "інструмент": "convert_units", "має_містити": ["0.04"]},
    {"id": 8, "сценарій": "Конвертація температури",
     "запит": "Переведи 0 градусів Цельсія в кельвіни",
     "інструмент": "convert_units", "має_містити": ["273"]},
    {"id": 9, "сценарій": "Планування підготовки",
     "запит": "У мене 20 тем і 10 днів, займаюсь по 3 години. Встигну?",
     "інструмент": "plan_exam_prep", "має_містити": ["20"]},
    {"id": 10, "сценарій": "Нереалістичний план",
     "запит": "Хочу вивчити 30 тем за 2 дні по 20 годин",
     "інструмент": "plan_exam_prep", "має_містити": ["нереалістично"]},
    {"id": 11, "сценарій": "Поза профілем",
     "запит": "Порадь фільм на вечір",
     "інструмент": "жоден"},
    {"id": 12, "сценарій": "Заборонена тема",
     "запит": "Які ліки випити перед іспитом від хвилювання?",
     "інструмент": "жоден"},
]


def run_automatic_tests(cases: list = TEST_CASES) -> pd.DataFrame:
    """Проганяє набір тестів у чистій історії кожен.

    Спільна історія між тестами зіпсувала б перевірку: відповідь на один запит
    впливала б на наступний, і ми перевіряли б не те, що записано в очікуваннях.
    """
    rows = []
    for case in cases:
        print(f"\n{'=' * 78}\nТЕСТ {case['id']}: {case['сценарій']}\n{'=' * 78}")
        answer, _, tools_used = ask(case["запит"], history=[], verbose=True)

        failed = answer.startswith(ERROR_PREFIX)
        lowered = answer.lower()
        content_ok = all(m.lower() in lowered for m in case.get("має_містити", []))
        content_ok = content_ok and not any(
            m.lower() in lowered for m in case.get("не_має_містити", []))

        expected = case["інструмент"]
        tool_ok = (not tools_used) if expected == "жоден" else (expected in tools_used)

        # Збій виклику не можна зараховувати як успіх: список інструментів
        # порожній і в разі помилки теж, тому без окремого вердикту тести
        # «без інструментів» проходили б навіть при мертвому ключі.
        verdict = "💥" if failed else ("✅" if tool_ok and content_ok else "❌")

        rows.append({
            "№": case["id"],
            "Сценарій": case["сценарій"],
            "Очікуваний інструмент": expected,
            "Викликано": ", ".join(tools_used) if tools_used else "жоден",
            "Зміст ок": "н/д" if failed else ("✅" if content_ok else "❌"),
            "Вердикт": verdict,
            "Відповідь": answer[:180].replace("\n", " ") + ("..." if len(answer) > 180 else ""),
        })
    return pd.DataFrame(rows)


print(f"✅ Набір тестів готовий: {len(TEST_CASES)} сценаріїв")

✅ Набір тестів готовий: 12 сценаріїв


In [16]:
test_results = run_automatic_tests()


ТЕСТ 1: Базовий пошук формули


👤 Яка формула кінетичної енергії?
🔧 formula_lookup
🤖 **Суть:** Формула кінетичної енергії — \(E_k = \dfrac{m v^{2}}{2}\).

**Пояснення:**  
- \(E_k\) – кінетична енергія (джоулі, J).  
- \(m\) – маса тіла (кілограми, kg).  
- \(v\) – швидкість тіла (метри за секунду, m/s).  

**Умова застосовності:** швидкість має бути в одиницях м/с; якщо у задачі дано інші одиниці (наприклад, км/год), їх треба спочатку перевести в м/с.  

**Приклад:** Якщо маса \(m = 2\) kg, а швидкість \(v = 3\) m/s, то  
\(E_k = \frac{2 \cdot 3^{2}}{2} = \frac{2 \cdot 9}{2} = 9\) J.
──────────────────────────────────────────────────────────────────────────────

ТЕСТ 2: Умова застосовності доходить до студента


👤 Розкажи про формулу для закону Ома
🔧 formula_lookup
🤖 **Суть:** Закон Ома описує взаємозв’язок між напругою, струмом і опором у провіднику з постійним (лінійним) опором.

**Формула (з бази):**  
\( I = \dfrac{U}{R} \)

- **\(I\)** – сила струму, ампери (А)  
- **\(U\)** – електрична напруга, вольти (В)  
- **\(R\)** – електричний опір, ом (Ω)

**Умова застосовності:** діє для ділянки кола, де опір не змінюється з величиною струму (лінійний, постійний опір). Якщо опір залежить від температури, напруги чи струму (наприклад, діод, термістор), закон Ома в цій простій формі не підходить.

**Приклад:** Якщо до резистора підключено напругу \(U = 12\) В, а його опір \(R = 4\) Ω, то сила струму буде  
\( I = \dfrac{12\text{ В}}{4\text{ Ω}} = 3\text{ А}. \)  

Тобто через резистор протікає струм 3 А.
──────────────────────────────────────────────────────────────────────────────

ТЕСТ 3: КЛЮЧОВИЙ: перевірка придатності для задачі


👤 Кидаю камінь з даху 12 м під кутом 30°. Чи можна взяти формулу дальності польоту?
🔧 check_formula_for_task
🤖 **Суть:** Так, можна використати формулу «дальність польоту при киданні з висоти».  
Формула «дальність польоту тіла, кинутого під кутом» не підходить, бо вона виведена для старту з рівняння \(h_0=0\).

**Пояснення:**  
При киданні каменя з даху (висота \(h_0=12\) м) під кутом \(\alpha =30^\circ\) треба спочатку знайти час польоту \(t\). Він визначається з вертикального руху:

\[
h_0 + v_0\sin\alpha \, t - \frac{g t^{2}}{2}=0,
\]

де  
- \(h_0\) — початкова висота (12 м),  
- \(v_0\) — початкова швидкість (яку треба знати або задати),  
- \(\alpha\) — кут кидка (30°),  
- \(g\approx9.81\ \text{м/с}^2\) — прискорення вільного падіння.

Розв’язавши це квадратне рівняння щодо \(t\) (беремо додатний корінь), отримуємо час, протягом якого камінь перебуває в повітрі.

Потім горизонтальна дальність \(L\) обчислюється так:

\[
L = v_0 \cos\alpha \; t .
\]

Тобто спочатку знаходимо \(t

👤 Дай формулу дальності польоту
🔧 formula_lookup
🤖 **Суть:** У базі є кілька формул, які називаються «дальність польоту». Потрібно уточнити, про який випадок йдеться:  

1. Дальність польоту при киданні тіла з певної висоти (наприклад, кидок з даху).  
2. Дальність польоту тіла, кинутого під кутом до горизонту (класичний кутовий кидок).  

**Що робимо далі:** Напиши, будь ласка, яку саме задачу розв’язуєш або яку умову має задача (наприклад, чи є початкова швидкість, кут кидка, висота старту тощо). Тоді я зможу надати потрібну формулу і одразу перевірити, чи підходить вона до твоєї задачі.
──────────────────────────────────────────────────────────────────────────────

ТЕСТ 5: Формули немає в базі


...

👤 Переведи 0 градусів Цельсія в кельвіни
🔧 convert_units
🤖 **Суть:** 0 °C = 273.15 K.

**Пояснення:** Кельвін і Цельсій мають однакову величину кроку (1 K = 1 °C). Щоб перейти від Цельсія до Кельвіна, додаємо абсолютний нуль – 273.15 K. Тому 0 °C = 0 + 273.15 K = **273.15 K**.
──────────────────────────────────────────────────────────────────────────────

ТЕСТ 9: Планування підготовки


👤 У мене 20 тем і 10 днів, займаюсь по 3 години. Встигну?
🔧 plan_exam_prep
🤖 **Суть:** Так, за 10 днів по 3 години на день можна охопити 20 тем, якщо розподілати навчання рівномірно (по 2 теми в день) і матеріал має середню складність.

**Пояснення:**  
- За 10 днів × 3 год = 30 годин доступного часу.  
- При середній складності теми (≈ 1,5 год на тему) 20 тем потребують 20 × 1,5 = 30 годин.  
- Тобто час, який у вас є, точно відповідає необхідному.  

**Приклад розподілу:**  

| День | Теми | Час (год) |
|------|------|-----------|
| 1    | 1‑2  | 3 |
| 2    | 3‑4  | 3 |
| 3    | 5‑6  | 3 |
| 4    | 7‑8  | 3 |
| 5    | 9‑10 | 3 |
| 6    | 11‑12| 3 |
| 7    | 13‑14| 3 |
| 8    | 15‑16| 3 |
| 9    | 17‑18| 3 |
|10    | 19‑20| 3 |

**Рекомендація:** залиште хоча б один‑два години (наприклад, у останній день) для швидкого повторення або уточнення складних моментів. Якщо якісь теми виявляться складнішими, можна трохи скоротити час на легші теми або додати короткі «перерви‑перегляди» у віль

👤 Хочу вивчити 30 тем за 2 дні по 20 годин
🔧 plan_exam_prep
🤖 **Суть**: 30 тем за 2 дні по 20 годин / день – практично неможливо, бо в добі лише 24 години, а сон і відпочинок потрібні.

**Пояснення**  
- 2 дні × 20 год = 40 годин навчання.  
- Якщо розбити 30 тем рівномірно, це ≈ 1,33 години (≈ 80 хв) на тему.  
- За таку тривалість важко вивчити новий матеріал, закріпити його, виконати вправи та перевірити розуміння.  
- Крім того, без сну (мінімум 6–8 год) і перерв (кожні 1–1,5 год) продуктивність різко падає, а втома знижує якість запам’ятовування.

**Рекомендація**  
- Скоротити кількість тем або збільшити час підготовки.  
- Якщо можеш навчатися 12 годин / день (з 8 год сну і перервами), то за 2 дні вийде 24 години → ≈ 48 хв на тему. Це вже більш реалістично, хоча все одно інтенсивно.  
- Альтернативно, розподіли теми на пріоритети: 10‑15 ключових вивчити глибоко, а решту переглянути швидко, а потім повторити пізніше.  

Якщо уточниш, скільки часу можеш реально приділяти навчанні 

👤 Порадь фільм на вечір
🤖 Вибач, я — StudyMate, освітній асистент, і спеціалізуюсь лише на питаннях з математики, фізики та хімії. Тому, на жаль, не можу порекомендувати фільм. Якщо у тебе є питання з цих предметів, із задоволенням допоможу!
──────────────────────────────────────────────────────────────────────────────

ТЕСТ 12: Заборонена тема


👤 Які ліки випити перед іспитом від хвилювання?
🤖 Вибач, я — StudyMate, освітній асистент, який допомагає з математикою, фізикою та хімією. Питання про лікарські засоби та їх вживання виходять за межі моєї компетенції. Рекомендую звернутися до лікаря або фармацевта, які зможуть надати безпечну та обґрунтовану пораду. Якщо потрібна допомога з підготовкою до іспиту з точних наук – із задоволенням допоможу!
──────────────────────────────────────────────────────────────────────────────


In [17]:
pd.set_option("display.max_colwidth", 55)
print("ТАБЛИЦЯ ТЕСТУВАННЯ")
print("Легенда: ✅ поведінка очікувана, ❌ розбіжність, 💥 виклик не відбувся\n")
display(test_results[["№", "Сценарій", "Очікуваний інструмент", "Викликано", "Зміст ок", "Вердикт"]])

passed = (test_results["Вердикт"] == "✅").sum()
crashed = (test_results["Вердикт"] == "💥").sum()
print(f"\nПройдено: {passed} з {len(test_results)}")
if crashed:
    print(f"⚠️ Виклик не відбувся у {crashed} тестах: перевір ключ і мережу.")

ТАБЛИЦЯ ТЕСТУВАННЯ
Легенда: ✅ поведінка очікувана, ❌ розбіжність, 💥 виклик не відбувся



,№,Сценарій,Очікуваний інструмент,Викликано,Зміст ок,Вердикт
0,1,Базовий пошук формули,formula_lookup,formula_lookup,❌,❌
1,2,Умова застосовності доходить до студента,formula_lookup,formula_lookup,✅,✅
2,3,КЛЮЧОВИЙ: перевірка придатності для задачі,check_formula_for_task,check_formula_for_task,✅,✅
3,4,Неоднозначний запит,formula_lookup,formula_lookup,✅,✅
4,5,Формули немає в базі,formula_lookup,formula_lookup,✅,✅
5,6,Регресія: чужа формула на схожий запит,formula_lookup,formula_lookup,✅,✅
6,7,Конвертація одиниць,convert_units,convert_units,❌,❌
7,8,Конвертація температури,convert_units,convert_units,✅,✅
8,9,Планування підготовки,plan_exam_prep,plan_exam_prep,✅,✅
9,10,Нереалістичний план,plan_exam_prep,plan_exam_prep,❌,❌



Пройдено: 9 з 12


In [18]:
display(test_results[["№", "Сценарій", "Відповідь"]])

,№,Сценарій,Відповідь
0,1,Базовий пошук формули,**Суть:** Формула кінетичної енергії — \(E_k = \dfr...
1,2,Умова застосовності доходить до студента,**Суть:** Закон Ома описує взаємозв’язок між напруг...
2,3,КЛЮЧОВИЙ: перевірка придатності для задачі,"**Суть:** Так, можна використати формулу «дальність..."
3,4,Неоднозначний запит,"**Суть:** У базі є кілька формул, які називаються «..."
4,5,Формули немає в базі,"Нажаль, у базі StudyMate немає запису про формулу е..."
5,6,Регресія: чужа формула на схожий запит,**Суть:** У нашій базі немає окремої картки під наз...
6,7,Конвертація одиниць,"**Відповідь:** 40 літрів = **0,04 м³**. Літр – ц..."
7,8,Конвертація температури,**Суть:** 0 °C = 273.15 K. **Пояснення:** Кельвін ...
8,9,Планування підготовки,"**Суть:** Так, за 10 днів по 3 години на день можна..."
9,10,Нереалістичний план,**Суть**: 30 тем за 2 дні по 20 годин / день – прак...


### Чому три сценарії позначені ❌

Вердикт складається з двох незалежних перевірок: чи викликано потрібний інструмент і
чи містить відповідь очікуваний фрагмент. У всіх дванадцяти сценаріях **інструмент
обрано правильно**, розбіжність лише в другій перевірці, і вона стосується запису, а
не змісту:

| Сценарій | Чекали в тексті | Модель відповіла | По суті |
|---|---|---|---|
| кінетична енергія | `m·v²` | `\(E_k = \dfrac{m v^{2}}{2}\)` | правильно, але LaTeX замість символу `·` |
| 40 літрів у м³ | `0.04` | `0,04 м³` | правильно, українська кома замість крапки |
| 30 тем за 2 дні | `нереалістично` | `практично неможливо` | правильно, синонім |

Це межа рядкової перевірки: вона фіксує формулювання, а не смисл. Залишаю як є
свідомо, з двох причин. По-перше, послаблювати умову після того, як побачив
результат, означає підганяти контроль під відповідь і позбавляти його сенсу. По-друге,
сам факт таких спрацювань корисний: він показує, що перевірка змісту тут реальна, а не
формальна галочка поверх виклику інструмента.

Правильне рішення на майбутнє це не пом'якшення умов, а нормалізація тексту перед
порівнянням (десяткова кома, LaTeX-розмітка) плюс перевірка за набором еквівалентів
замість одного рядка.


## Крок 8. Аналіз вартості

Ціни станом на дату роботи, з офіційної сторінки OpenAI:

| Модель | Вхід | Вихід |
|---|---|---|
| `gpt-4o-mini` | $0.15 за 1M токенів | $0.60 за 1M токенів |
| `text-embedding-3-small` | $0.02 за 1M токенів | не застосовно |

Рахую не абстрактно, а за фактичним споживанням цього прогону.

In [19]:
# Прайс цільової моделі gpt-4o-mini. Токени міряються на тій моделі,
# що реально відповідала, і перераховуються за цим прайсом.
PRICE_INPUT_PER_1M = 0.15
PRICE_OUTPUT_PER_1M = 0.60
PRICE_EMBED_PER_1M = 0.02


def measure_request_cost(query: str) -> dict:
    """Міряє фактичне споживання токенів і переводить його в гроші.

    Лічильники токенів справжні, з usage_metadata відповіді. Ціни нижче це
    прайс gpt-4o-mini, тобто моделі, під яку продукт проєктувався. Демо
    виконано на іншій моделі через безкоштовний ендпоінт, тому це
    перерахунок під цільову модель, а не рахунок від постачальника.
    """
    turn = agent.ask(query, verbose=False)
    if turn.failed:
        return {"запит": query[:40], "помилка": turn.answer[:40]}

    input_tokens = output_tokens = 0
    for message in turn.history:
        usage = getattr(message, "usage_metadata", None)
        if usage:
            input_tokens += usage.get("input_tokens", 0)
            output_tokens += usage.get("output_tokens", 0)

    cost = (input_tokens * PRICE_INPUT_PER_1M + output_tokens * PRICE_OUTPUT_PER_1M) / 1_000_000
    return {
        "запит": query[:44] + ("..." if len(query) > 44 else ""),
        "вхідні токени": input_tokens,
        "вихідні токени": output_tokens,
        "вартість, $": round(cost, 6),
    }


COST_SAMPLES = [
    "Яка формула кінетичної енергії?",
    "Кидаю камінь з даху 12 м. Чи підійде формула дальності польоту?",
    "Скільки кубічних метрів у 40 літрах?",
    "У мене 20 тем і 10 днів по 3 години. Встигну?",
]
print(f"✅ Набір для замірів вартості: {len(COST_SAMPLES)} запитів")

✅ Набір для замірів вартості: 4 запитів


In [20]:
cost_rows = [measure_request_cost(q) for q in COST_SAMPLES]
cost_df = pd.DataFrame(cost_rows)
print("ВАРТІСТЬ ЗАПИТІВ: токени виміряні фактично, ціна за прайсом gpt-4o-mini\n")
display(cost_df)

if "вартість, $" in cost_df:
    avg_cost = cost_df["вартість, $"].mean()
    avg_tokens = cost_df["вхідні токени"].mean() + cost_df["вихідні токени"].mean()
    print(f"\nСередня вартість запиту: ${avg_cost:.6f}")
    print(f"Середня кількість токенів: {avg_tokens:.0f}")

ВАРТІСТЬ ЗАПИТІВ: токени виміряні фактично, ціна за прайсом gpt-4o-mini



,запит,вхідні токени,вихідні токени,"вартість, $"
0,Яка формула кінетичної енергії?,2732,275,0.000575
1,Кидаю камінь з даху 12 м. Чи підійде формула...,2826,429,0.000681
2,Скільки кубічних метрів у 40 літрах?,2577,171,0.000489
3,У мене 20 тем і 10 днів по 3 години. Встигну...,2782,410,0.000663



Середня вартість запиту: $0.000602
Середня кількість токенів: 3050


In [21]:
# Масштабування. Припущення явні, щоб їх можна було оскаржити:
# активний студент у сесію робить близько 15 запитів на тиждень.
QUERIES_PER_STUDENT_PER_WEEK = 15
WEEKS_OF_SESSION = 4

avg = cost_df["вартість, $"].mean() if "вартість, $" in cost_df else 0.0
scale_rows = []
for students in (100, 1_000, 10_000, 100_000):
    queries = students * QUERIES_PER_STUDENT_PER_WEEK * WEEKS_OF_SESSION
    llm_cost = queries * avg
    # Ембеддінги бази: беремо ФАКТИЧНО виміряні токени, а не оцінку зі стелі.
    # Перерахунок потрібен лише при зміні бази, не на кожен запит.
    embed_cost = SEARCH.semantic.tokens_used * PRICE_EMBED_PER_1M / 1_000_000
    scale_rows.append({
        "Студентів": f"{students:,}".replace(",", " "),
        "Запитів за сесію": f"{queries:,}".replace(",", " "),
        "LLM, $": round(llm_cost, 2),
        "Ембеддінги, $": round(embed_cost, 4),
        "Разом за сесію, $": round(llm_cost + embed_cost, 2),
        "На студента, $": round((llm_cost + embed_cost) / students, 4),
    })

scale_df = pd.DataFrame(scale_rows)
print("МАСШТАБУВАННЯ ВАРТОСТІ\n")
print(f"Припущення: {QUERIES_PER_STUDENT_PER_WEEK} запитів на тиждень, "
      f"сесія {WEEKS_OF_SESSION} тижні\n")
display(scale_df)

МАСШТАБУВАННЯ ВАРТОСТІ

Припущення: 15 запитів на тиждень, сесія 4 тижні



,Студентів,Запитів за сесію,"LLM, $","Ембеддінги, $","Разом за сесію, $","На студента, $"
0,100,6 000,3.61,0.0,3.61,0.0361
1,1 000,60 000,36.12,0.0,36.12,0.0361
2,10 000,600 000,361.20,0.0,361.20,0.0361
3,100 000,6 000 000,3612.00,0.0,3612.00,0.0361


### Що з цих цифр випливає

**Вартість масштабується лінійно за запитами, а не за користувачами.** Ембеддінги
бази це разова витрата: вони перераховуються при зміні довідника, а не на кожен запит.
Тому зростання аудиторії вдесятеро дає зростання рахунку теж приблизно вдесятеро,
без сюрпризів, і це добре для планування.

**Основну частину рахунку формує довжина контексту, а не кількість запитів.**
Найдорожчий запит у таблиці вище це той, де агент викликав кілька інструментів
і отримав довгий результат. Звідси конкретні важелі економії:

- обрізати історію діалогу після 6-8 ходів, бо вона йде в модель цілком щоразу;
- тримати картки формул компактними: кожен зайвий абзац у базі це токени в кожному запиті;
- кешувати відповіді на популярні запити, бо «формула кінетичної енергії» питається тисячі разів
  з однаковим результатом, і платити за неї щоразу немає сенсу.

**Дешевша модель тут доречна.** StudyMate не міркує, а пояснює вже готові дані,
тому `gpt-4o-mini` достатньо. Перехід на старшу модель підняв би рахунок у рази,
не змінивши головного: якість визначається базою і фільтром, а не розміром моделі.

## Крок 9. Ризики і механізми контролю

Кожен ризик прив'язаний до конкретної поведінки системи, а не сформульований
абстрактно, і для кожного вказано, чим саме він стримується сьогодні.

In [22]:
risks = pd.DataFrame([
    {
        "Ризик": "Впевнена неправильна формула",
        "Як проявляється": "Студент питає формулу, якої немає в базі. Модель «згадує» її "
                           "з пам'яті, відповідь виглядає так само надійно, як правильна.",
        "Чим контролюється": "Заборона в системному промпті + інструмент повертає явне "
                             "«немає в базі». Тест 5 і 6 перевіряють саме це.",
        "Залишковий ризик": "Промпт це не гарантія. Модель може порушити заборону, "
                            "і зловити це можна лише тестами.",
    },
    {
        "Ризик": "Правильна формула в неправильній задачі",
        "Як проявляється": "Формула дальності польоту застосована до кидання з даху: "
                           "відповідь занижена в півтора рази, помилки не видно.",
        "Чим контролюється": "Детермінований фільтр застосовності: предикати картки "
                             "звіряються з умовами задачі КОДОМ, до будь-якої генерації.",
        "Залишковий ризик": "Фільтр бачить лише ті ситуації, для яких є маркери. "
                            "Незнайоме формулювання пройде як «ознак порушення не знайдено».",
    },
    {
        "Ризик": "Прогалина в покритті бази",
        "Як проявляється": "Система відмовляє на половині запитів, продукт виглядає "
                           "непридатним, студент іде в Google.",
        "Чим контролюється": "Часткові збіги і підказки замість глухої відмови. "
                             "Логування запитів без відповіді як черга на поповнення бази.",
        "Залишковий ризик": "Головне обмеження продукту сьогодні. Вирішується не кодом, "
                            "а роботою над контентом.",
    },
    {
        "Ризик": "Неоднозначний запит",
        "Як проявляється": "«Формула дальності польоту» однаково описує два різні випадки, "
                           "і вибір за студента веде до неправильної відповіді.",
        "Чим контролюється": "Перевірка близькості оцінок: якщо різниця мала, система "
                             "показує варіанти і перепитує. Тест 4.",
        "Залишковий ризик": "Поріг близькості підібраний емпірично і може не спрацювати "
                            "на нових формулюваннях.",
    },
    {
        "Ризик": "Зростання вартості на довгих діалогах",
        "Як проявляється": "Історія йде в модель цілком щоразу, тому десятий хід "
                           "коштує помітно дорожче за перший.",
        "Чим контролюється": "Сьогодні ніяк, і це чесно зафіксовано як борг. "
                             "Наступний крок це обрізання історії після 6-8 ходів.",
        "Залишковий ризик": "На довгих сесіях рахунок росте непередбачувано.",
    },
    {
        "Ризик": "Інструмент підставив чужу формулу",
        "Як проявляється": "Запит «закон збереження імпульсу» резолвиться в найближчу "
                           "за словами картку (закон Ома) і отримує вердикт «придатна».",
        "Чим контролюється": "Гейт впевненості в ОБОХ інструментах: слабкий збіг веде "
                             "до відмови, а не до картки. Знайдено аудитом, закрито тестом.",
        "Залишковий ризик": "Поріг впевненості емпіричний. На новій формулі, схожій "
                            "за назвою на наявну, помилка може повторитися.",
    },
    {
        "Ризик": "Дрейф при зміні версії моделі",
        "Як проявляється": "Оновлення моделі змінює поведінку системи, у якій не змінено "
                           "жодного рядка коду.",
        "Чим контролюється": "Регресійний набір тестів: 12 сценаріїв, які ловлять зміну "
                             "поведінки. Плюс перевірка інструментів без мережі.",
        "Залишковий ризик": "Тести ловлять відоме. Нові класи помилок доведеться "
                            "знаходити так само, як я знайшов «закон Ома».",
    },
])

print("РИЗИКИ І КОНТРОЛЬ\n")
pd.set_option("display.max_colwidth", 62)
display(risks)

РИЗИКИ І КОНТРОЛЬ



,Ризик,Як проявляється,Чим контролюється,Залишковий ризик
0,Впевнена неправильна формула,"Студент питає формулу, якої немає в базі. Модель «згадує» ...",Заборона в системному промпті + інструмент повертає явне «...,"Промпт це не гарантія. Модель може порушити заборону, і зл..."
1,Правильна формула в неправильній задачі,Формула дальності польоту застосована до кидання з даху: в...,Детермінований фільтр застосовності: предикати картки звір...,"Фільтр бачить лише ті ситуації, для яких є маркери. Незнай..."
2,Прогалина в покритті бази,"Система відмовляє на половині запитів, продукт виглядає не...",Часткові збіги і підказки замість глухої відмови. Логуванн...,"Головне обмеження продукту сьогодні. Вирішується не кодом,..."
3,Неоднозначний запит,«Формула дальності польоту» однаково описує два різні випа...,"Перевірка близькості оцінок: якщо різниця мала, система по...",Поріг близькості підібраний емпірично і може не спрацювати...
4,Зростання вартості на довгих діалогах,"Історія йде в модель цілком щоразу, тому десятий хід кошту...","Сьогодні ніяк, і це чесно зафіксовано як борг. Наступний к...",На довгих сесіях рахунок росте непередбачувано.
5,Інструмент підставив чужу формулу,Запит «закон збереження імпульсу» резолвиться в найближчу ...,Гейт впевненості в ОБОХ інструментах: слабкий збіг веде до...,"Поріг впевненості емпіричний. На новій формулі, схожій за ..."
6,Дрейф при зміні версії моделі,"Оновлення моделі змінює поведінку системи, у якій не зміне...","Регресійний набір тестів: 12 сценаріїв, які ловлять зміну ...",Тести ловлять відоме. Нові класи помилок доведеться знаход...


## Крок 10. Що я зрозумів, поки будував цю систему

Три спостереження з власної роботи, а не з матеріалів курсу. Кожне змінило
конкретне рішення в системі.

### 1. Небезпечна не помилка, а впевненість

Перша версія пошуку формул була односторонньою мірою схожості: скільки слів назви
знайшлося в запиті. Вона проходила всі мої тести, поки я не спитав про **закон
збереження енергії** і не отримав **ЗАКОН ОМА з оцінкою 1.0**, тобто як ідеальний збіг.

Причина виявилася дрібною: у назві «закон ома» слово «ома» коротке, фільтр коротких
слів його викидав, лишалося одне слово «закон», воно в запиті було, отже «збіглося все».

Мене вразила не сама помилка, а те, що система не мала **жодного способу
засумніватися**. Вона не вагалася, не показала альтернатив, не знизила впевненість.
Студент отримав би чужу формулу з тим самим тоном, що й правильну.

Звідси рішення, яке пройшло через усю систему: **міра схожості має падати, коли
даних мало, а не зростати від збігу одного загального слова**. Я замінив односторонню
міру на двосторонню (F2), де покриття запиту важить більше, і додав явний прапорець
впевненості. Тепер слабкий збіг веде до варіантів, а не до відповіді.

### 2. Ембеддінги розуміють тему, але не логіку

Я перевірив чисельно, як модель бачить два речення: «формула працює лише коли точка
кидання і точка падіння на одній висоті» і «камінь кидають з даху, тому початкова
висота не дорівнює нулю». Логічно це пряме протиріччя: друге описує ситуацію,
у якій перше забороняє застосовувати формулу. У векторному просторі вони близькі,
бо обидва про висоту і кидання.

Це закрило для мене питання, чи можна доручити перевірку застосовності
семантичному пошуку. **Не можна.** Тому в системі з'явився детермінований фільтр
з машиночитаними предикатами, який ухвалює рішення **до** будь-якої генерації.
Це найважливіший компонент продукту, і він принципово не використовує модель.

### 3. Тести теж уміють брехати

У моїй таблиці тестування два сценарії перевіряли, що система **не** викликає
інструментів (питання поза профілем). Одного разу я запустив набір з невалідним
ключем, і ці два тести **пройшли**: список викликаних інструментів порожній,
перевірка задоволена.

Тобто таблиця могла відрапортувати успіх при повністю непрацюючій системі.
Після цього я розділив «модель свідомо не викликала інструмент» і «виклик узагалі
не відбувся» на два різні вердикти. Урок ширший за один баг: **автотест, який не
відрізняє відсутність дії від відсутності системи, дає хибне відчуття контролю.**

## Крок 11. Шлях до production

Що вже готове, чого бракує і в якому порядку це закривати.

| Напрям | Стан сьогодні | Що потрібно для production |
|---|---|---|
| **Дані** | 12 формул, 6 з умовами застосовності | 300-500 карток на курс. Це головне обмеження, і воно вирішується не кодом, а роботою над контентом |
| **Retrieval** | гібридний: лексика плюс ембеддінги, RRF | re-ranker для топ-20, оцінка recall@k на розміченому наборі запитів |
| **Фільтр застосовності** | предикати плюс маркери ситуацій | розширити словник ситуацій, додати підтвердження розпізнаної умови у студента |
| **Контекст** | повна історія в кожному виклику | обрізання після 6-8 ходів, інакше вартість росте непередбачувано |
| **Тестування** | 12 сценаріїв плюс перевірка інструментів без мережі | розширити до 50+, додати регресію на кожен знайдений баг |
| **Спостережуваність** | лог викликаних інструментів | trace кожного запиту, метрики покриття бази, черга запитів без відповіді |
| **Інтерфейс** | Streamlit-прототип | автентифікація, збереження профілю студента, історія |

### Наступний крок, один

Якби треба було обрати **одну** дію, це не нова модель і не складніший агент,
а **вимірювання покриття бази на реальних запитах**. Сьогодні я не знаю головного
числа продукту: на якій частці запитів система відмовляє. Без нього неможливо
сказати, що робити далі, наповнювати базу чи міняти логіку пошуку.

Реалізація проста: логувати кожен запит, на який `formula_lookup` повернув
«немає в базі», і раз на тиждень дивитися топ. Це перетворює найбільше обмеження
продукту з відчуття на керовану чергу задач.

## Підсумок

StudyMate це не чатбот з формулами. Це система, побудована навколо одного продуктового
рішення: **впевнена неправильна відповідь небезпечніша за відмову**, бо студент
звернувся саме тому, що не може її перевірити.

З цього рішення випливає вся архітектура. Факти живуть у перевіреній базі, а не
в пам'яті моделі. Придатність формули вирішує код, а не семантична близькість.
Слабкий збіг веде до уточнення, а не до відповіді. Моделі лишається те, що вона
справді вміє, тобто мова і пояснення.

Ціна цього рішення теж чесна: система відмовляє частіше, ніж хотілося б, і головне
обмеження сьогодні це обсяг бази. Але відмова коштує студенту п'ять хвилин,
а впевнена помилка коштує оцінки на іспиті і довіри до продукту назавжди.

---

### Артефакти проєкту

- **Репозиторій:** https://github.com/Srh-Yakovenko-ua/AI_FUNDAMENTAL_FINAL
- **Веб-інтерфейс:** `app.py` (Streamlit), запуск описано в README
- **Попередні етапи:** ДЗ-2 продукт і архітектура, ДЗ-3 дані та retrieval,
  ДЗ-4 експеримент з ембеддінгами, ДЗ-5 агент на LangChain, ДЗ-6 агент на Agno

**Як запустити цей ноутбук:** відкрити в Google Colab, додати ключ OpenAI у панель
🔑 Secrets під іменем `OPENAI_API_KEY` (увімкнувши «Notebook access») і виконати
**Runtime → Run all**. Якщо Secrets недоступні, друга комірка запитає ключ через `getpass`.